# EPA – data exploration

Quick exploratory pass over the candidate-level examples derived from the
reference EPA dataset at <https://github.com/RevRameshkumar/EPADataset>.
Run this *after* `python -m src.data.download && python -m src.data.build_examples`.

We focus on:
1. dataset shape and class balance
2. inter-annotator agreement and consensus mode
3. positive-rate by candidate role (sender / To / Cc) — a key signal for the model
4. positive-rate by candidate-set size (single vs multi recipient — the paper's primary cut)
5. quick look at task-text length distributions
6. addressee-tagging cues: explicit name in task, implicit *you*/*your*

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from src.utils.io import load_config

cfg = load_config('../config.yaml')
df = pd.read_csv('../' + cfg['paths']['examples_csv'])
print('candidate rows:', len(df))
print('positives:    ', int(df.label.sum()))
print('class prior:  ', round(df.label.mean(), 4))
print('unique HITs:  ', df['email_task_id'].nunique())
df.head()

In [ ]:
# Inter-annotator agreement: how many HITs had perfect agreement vs only majority?
hit_level = df.drop_duplicates('email_task_id')[['email_task_id', 'n_judges', 'perfect_agreement', 'no_one_responsible']]
print('HIT counts by # of judges:')
print(hit_level['n_judges'].value_counts().sort_index())
print()
print(f"Perfect-agreement HITs : {int(hit_level['perfect_agreement'].sum())} / {len(hit_level)}")
print(f"`no-one responsible`   : {int(hit_level['no_one_responsible'].sum())}")

In [ ]:
# Class balance per candidate role. The huge gap between sender (~1.7%) and
# To (~87%) is the dominant structural signal for the model.
df.groupby('candidate_role')['label'].agg(['count', 'sum', 'mean']).rename(columns={'mean': 'positive_rate'})

In [ ]:
# Positive rate by candidate-set size — the paper notes multi-recipient is harder.
df.groupby('num_total_candidates')['label'].agg(['count', 'mean']).rename(columns={'mean': 'positive_rate'})

In [ ]:
# Task length distribution.
df['task_tokens'] = df['task'].fillna('').str.split().str.len()
df['body_tokens'] = df['body'].fillna('').str.split().str.len()
df[['task_tokens', 'body_tokens']].describe(percentiles=[0.5, 0.9, 0.99])

In [ ]:
# Implicit-you cases: tasks containing 'you' / 'your'.
import re
has_you = df['task'].fillna('').str.contains(r'\byou\b|\byour\b', regex=True, case=False)
df.assign(has_you=has_you).groupby('has_you')['label'].agg(['count', 'mean']).rename(columns={'mean': 'positive_rate'})

In [ ]:
# How often does the candidate's first name appear in the task? When it does, the positive rate jumps.
first = df['candidate_name'].fillna('').str.split().str[0].fillna('').str.lower()
task_lower = df['task'].fillna('').str.lower()
first_name_in_task = [bool(f) and (f in re.split(r'\W+', t)) for f, t in zip(first, task_lower)]
df.assign(first_name_in_task=first_name_in_task).groupby('first_name_in_task')['label'].agg(['count', 'mean']).rename(columns={'mean': 'positive_rate'})

In [ ]:
# Candidate-set size distribution — useful when reasoning about multi-recipient over-assignment.
df.drop_duplicates('email_task_id')['num_total_candidates'].value_counts().sort_index()